# 03 — Model Comparison: Addiction Classifier Benchmark

**Purpose:** Compare classifier architectures on the UCI EEG dataset.
Target: reproduce ~95% accuracy (published CNN baseline).

**Models:** SVM, Random Forest, EEGNet, DSCnet

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from classifiers.addiction.data import load_uci_dataset, AddictionConfig, split_by_subject
from classifiers.addiction.train import train_and_evaluate, PUBLISHED_BASELINES
from classifiers.addiction.features import AddictionFeatureConfig

plt.style.use('dark_background')
UCI_DIR = Path('../../data/raw/uci_eeg')

## 1. Load and Split Data

In [ ]:
config = AddictionConfig(data_dir=UCI_DIR)
dataset = load_uci_dataset(config)
split = split_by_subject(dataset, config)

print(f'Train: {len(split.train_subjects)} subjects, {len(split.train.labels)} trials')
print(f'Val: {len(split.val_subjects)} subjects, {len(split.val.labels)} trials')
print(f'Test: {len(split.test_subjects)} subjects, {len(split.test.labels)} trials')

## 2. Train All Models

In [ ]:
feature_config = AddictionFeatureConfig()
results = {}

for model_type in ['svm', 'random_forest']:
    print(f'\nTraining {model_type}...')
    model, metrics, preds = train_and_evaluate(
        split.train, split.test, model_type=model_type,
        feature_config=feature_config
    )
    results[model_type.upper()] = metrics
    print(f'  Acc: {metrics.accuracy:.4f}, AUC: {metrics.auc:.4f}, F1: {metrics.f1:.4f}')

for model_type in ['eegnet', 'dscnet']:
    print(f'\nTraining {model_type}...')
    try:
        model, metrics, preds = train_and_evaluate(
            split.train, split.test, model_type=model_type,
            model_kwargs={'n_epochs': 50}
        )
        results[model_type.upper()] = metrics
        print(f'  Acc: {metrics.accuracy:.4f}, AUC: {metrics.auc:.4f}, F1: {metrics.f1:.4f}')
    except Exception as e:
        print(f'  Failed: {e}')

## 3. Results Comparison

In [ ]:
# Print table
baseline = PUBLISHED_BASELINES['cnn_uci']
print(f"\n{'='*65}")
print(f"{'Model':<15} {'Accuracy':>10} {'AUC':>10} {'F1':>10}")
print(f"{'-'*65}")
print(f"{'CNN (pub.)*':<15} {baseline['accuracy']:>10.4f} {baseline['auc']:>10.4f} {baseline['f1']:>10.4f}")
print(f"{'-'*65}")
for name, m in results.items():
    print(f'{name:<15} {m.accuracy:>10.4f} {m.auc:>10.4f} {m.f1:>10.4f}')
print(f"{'='*65}")

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, metric, label in zip(axes, ['accuracy', 'auc', 'f1'], ['Accuracy', 'AUC', 'F1']):
    names = list(results.keys()) + ['CNN\n(baseline)']
    vals = [getattr(results[m], metric) for m in results] + [baseline[metric]]
    colors = ['#4FC3F7'] * len(results) + ['#999']
    bars = ax.bar(names, vals, color=colors, edgecolor='#333', linewidth=2)
    ax.set_ylabel(label)
    ax.set_ylim(0, 1)
    ax.axhline(y=baseline[metric], color='#FFD54F', linestyle='--', linewidth=1, alpha=0.7)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontsize=9)
plt.suptitle('Addiction Classifier — Model Comparison', fontsize=13)
plt.tight_layout()
plt.show()